# Notebook 3: Data Cleaning and Transformation

This notebook focuses on cleaning and transforming the Developer Survey dataset to prepare it for analysis and visualization in PowerBI. Building on the exploratory data analysis (EDA) from Notebook 2, we apply necessary and sensible transformations to a subset of relevant columns, addressing issues such as null values, inconsistent formats, and duplicates. The cleaned dataset will be loaded into a new PostgreSQL table called `clean_survey`, optimized for downstream use.

### Objectives
- Clean and standardize data in the specified relevant columns.
- Handle missing values and duplicates appropriately.
- Ensure data types and formats are suitable for PowerBI.
- Create and populate the `clean_survey` table in PostgreSQL.

## Setup

We import the required libraries for data manipulation and database interaction, then establish a connection to the PostgreSQL database using environment variables for security.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String, Float, Boolean
from sqlalchemy_utils import database_exists, create_database
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Get environment variables
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

# Build database connection URL
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Create the database engine
engine = create_engine(DATABASE_URL)
print('Database connection established successfully.')

Database connection established successfully.


## Data Loading

We load the dataset from the `raw_survey` table into a Pandas DataFrame for processing.

In [2]:
# Load data from raw_survey table
query = "SELECT * FROM raw_survey"
try:
    df = pd.read_sql(query, engine)
    print(f"Data loaded successfully. Rows: {len(df)}, Columns: {len(df.columns)}")
except Exception as e:
    print(f"Error loading data: {e}")
    raise

Data loaded successfully. Rows: 98855, Columns: 129


In [3]:
# Tasa de conversión a COP aproximada (2018)
conversion_rates = {
    'U.S. dollars ($)': 2900,
    'Euros (€)': 3400,
    'Indian rupees (₹)': 40,
    'British pounds sterling (£)': 3800,
    'Canadian dollars (C$)': 2200,
    'Russian rubles (₽)': 50,
    'Brazilian reais (R$)': 800,
    'Australian dollars (A$)': 2200,
    'Polish złoty (zł)': 800,
    'Swedish kroner (SEK)': 350,
    'Swiss francs': 3100,
    'Chinese yuan renminbi (¥)': 420,
    'Danish krone (kr)': 500,
    'Mexican pesos (MXN$)': 150,
    'South African rands (R)': 220,
    'Norwegian krone (kr)': 400,
    'Singapore dollars (S$)': 2100,
    'Japanese yen (¥)': 25,
    'Bitcoin (btc)': 0
}

# Asegurar tipo numérico en Salary
df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')

# Crear máscara de registros válidos
valid_mask = (
    (df['Salary'] > 0) &
    (df['SalaryType'].isin(['Monthly', 'Yearly', 'Weekly'])) &
    (df['Currency'].isin(conversion_rates.keys()))
)

# Procesar solo los válidos
df_valid = df[valid_mask].copy()

# Conversión a salario mensual
def convertir_salario_mensual(row):
    if row['SalaryType'] == 'Yearly':
        return row['Salary'] / 12
    elif row['SalaryType'] == 'Weekly':
        return row['Salary'] * 4
    else:
        return row['Salary']

df_valid['MonthlySalary'] = df_valid.apply(convertir_salario_mensual, axis=1)

# Conversión a COP
def convertir_a_cop(row):
    tasa = conversion_rates.get(row['Currency'], 0)
    return round(row['MonthlySalary'] * tasa, 2) if tasa > 0 else 0

df_valid['StandardizedMonthlySalaryCOP'] = df_valid.apply(convertir_a_cop, axis=1)

# Eliminar salarios exageradamente altos
df_valid.loc[df_valid['StandardizedMonthlySalaryCOP'] > 200_000_000, 'StandardizedMonthlySalaryCOP'] = 0

# Inicializar la columna con ceros en todo el DataFrame
df['StandardizedMonthlySalaryCOP'] = 0

# Asignar los valores calculados a los registros válidos
df.loc[df_valid.index, 'StandardizedMonthlySalaryCOP'] = df_valid['StandardizedMonthlySalaryCOP']

C:\Users\Nicolas Cuaran\AppData\Local\Temp\ipykernel_7276\325669579.py:62: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[16150000.    4766666.67 29000000.   ... 22166666.67   414000.
  2380000.  ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[df_valid.index, 'StandardizedMonthlySalaryCOP'] = df_valid['StandardizedMonthlySalaryCOP']


In [4]:
# Paso 1: Lista de columnas relevantes
benefit_cols = [
    'AssessBenefits2', 'AssessBenefits3', 'AssessBenefits4', 'AssessBenefits5',
    'AssessBenefits7', 'AssessBenefits8', 'AssessBenefits9',
    'AssessBenefits10', 'AssessBenefits11'
]

# Paso 2: Asegurar tipo entero en cada columna (con soporte para nulos)
df[benefit_cols] = df[benefit_cols].apply(pd.to_numeric, errors='coerce').astype('Int64')

# Paso 3: Calcular promedio por fila y redondear al entero más cercano
df['AvgNonSalaryBenefitsImportance'] = df[benefit_cols].mean(axis=1).round().astype('Int64')

# Paso 4: Verifica
print(df['AvgNonSalaryBenefitsImportance'].value_counts().sort_index())



AvgNonSalaryBenefitsImportance
5     1105
6    29460
7    34353
Name: count, dtype: Int64


In [5]:
# Paso 1: Definir columnas de interés
contacto_fuera_horario_cols = [
    'JobEmailPriorities1',
    'JobEmailPriorities2',
    'JobEmailPriorities3',
    'JobEmailPriorities6',
    'JobEmailPriorities7'
]

# Paso 2: Asegurar que todos los valores sean enteros válidos
df[contacto_fuera_horario_cols] = df[contacto_fuera_horario_cols].apply(pd.to_numeric, errors='coerce').astype('Int64')

# Paso 3: Calcular promedio por fila, redondear al entero más cercano
df['AvgAltContactImportance'] = df[contacto_fuera_horario_cols].mean(axis=1).round().astype('Int64')

# Paso 4: Verifica conteo de valores
print(df['AvgAltContactImportance'].value_counts().sort_index())


AvgAltContactImportance
3     3058
4    24265
5    18890
Name: count, dtype: Int64


In [6]:
# Paso 1: Definir columnas relacionadas con el lado administrativo
admin_interest_cols = [
    'AssessJob2', 'AssessJob3', 'AssessJob5', 'AssessJob6',
    'AssessJob8', 'AssessJob9', 'AssessJob10'
]

# Paso 2: Asegurar que los valores sean numéricos y enteros
df[admin_interest_cols] = df[admin_interest_cols].apply(pd.to_numeric, errors='coerce').astype('Int64')

# Paso 3: Calcular la media por fila, redondearla, y crear la nueva columna
df['AvgAdminInterest'] = df[admin_interest_cols].mean(axis=1).round().astype('Int64')

# Paso 4: Verificar la distribución
print(df['AvgAdminInterest'].value_counts().sort_index())

AvgAdminInterest
4     2562
5    28482
6    32205
7     3736
Name: count, dtype: Int64


In [7]:
# Diccionario de mapeo para renombrar columnas específicas
rename_columns = {
    'AssessBenefits1': 'Benefit_SalaryBonuses',
    'AssessBenefits6': 'Benefit_RetirementPlan',
    'AssessJob1': 'Importance_Industry',
    'AssessJob7': 'Importance_RemoteWork',
    'AssessJob4': 'Importance_Technologies'
}

# Renombrar en el DataFrame
df.rename(columns=rename_columns, inplace=True)

## Column Selection

We filter the dataset to include only the relevant columns specified for analysis and visualization in PowerBI.

In [8]:
relevant_columns = [
    # Demographic
    'Country', 'Gender', 'Age', 'FormalEducation', 'RaceEthnicity',
    # Professional Experience
    'DevType', 'CompanySize', 'Employment', 'YearsCodingProf','UndergradMajor', 'UpdateCV',
    # Technologies and Tools
    'LanguageWorkedWith',  'DatabaseWorkedWith', 'PlatformWorkedWith',
    'FrameworkWorkedWith', 'IDE', 'OperatingSystem', 'CommunicationTools', 'Methodology', 'VersionControl',
    # Job Satisfaction
    'JobSatisfaction', 'CareerSatisfaction', 'HopeFiveYears', 'AvgAdminInterest', 'AvgAltContactImportance', 'AvgNonSalaryBenefitsImportance',
    # Miscellaneous
    'OpenSource', 'StackOverflowVisit', 'StackOverflowHasAccount',
    'StackOverflowParticipate', 'AIDangerous', 'AIInteresting',
    'AIResponsible', 'AIFuture', 'Hobby',
    # Fact table
    'Benefit_SalaryBonuses', 'Benefit_RetirementPlan', 'Importance_Industry',
    'Importance_RemoteWork', 'Importance_Technologies', 'StandardizedMonthlySalaryCOP',
]

df_relevant = df[relevant_columns]
print("New DataFrame with relevant columns created. Shape:", df_relevant.shape)

New DataFrame with relevant columns created. Shape: (98855, 41)


## Null Handling

Most nulls have been addressed in the transformations above. We verify the remaining null counts to ensure completeness.

In [9]:
# Check for remaining nulls
null_counts = df_relevant.isnull().sum().head(41)
print('Remaining null counts:\n', null_counts[null_counts > 0])

Remaining null counts:
 Country                             412
Gender                            34386
Age                               34281
FormalEducation                    4152
RaceEthnicity                     41382
DevType                            6757
CompanySize                       27324
Employment                         3534
YearsCodingProf                   20952
UndergradMajor                    19819
UpdateCV                          33316
LanguageWorkedWith                20521
DatabaseWorkedWith                32585
PlatformWorkedWith                32856
FrameworkWorkedWith               47235
IDE                               23457
OperatingSystem                   22676
CommunicationTools                41885
Methodology                       39874
VersionControl                    24557
JobSatisfaction                   29579
CareerSatisfaction                22351
HopeFiveYears                     23137
AvgAdminInterest                  31870
AvgAltContactImp

In [10]:
# Diccionario actualizado con valores en inglés y capitalización coherente
na_replacements = {
    'Country': 'Other',
    'Gender': 'Other',
    'Age': 'Unknown',
    'FormalEducation': 'Unknown',
    'RaceEthnicity': 'Other',
    'DevType': 'Other',
    'CompanySize': 'Other',
    'Employment': 'Prefer not to say',
    'YearsCodingProf': 'Unknown',
    'UndergradMajor': 'Prefer not to say',
    'UpdateCV': 'Prefer not to say',
    'LanguageWorkedWith': 'Other',
    'DatabaseWorkedWith': 'Other',
    'PlatformWorkedWith': 'Other',
    'FrameworkWorkedWith': 'Other',
    'IDE': 'Other',
    'OperatingSystem': 'Other',
    'CommunicationTools': 'Other',
    'Methodology': 'Other',
    'VersionControl': 'Other',
    'JobSatisfaction': 'Prefer not to say',
    'CareerSatisfaction': 'Prefer not to say',
    'HopeFiveYears': 'Other',
    'AvgAdminInterest': 99,
    'AvgAltContactImportance': 99,
    'AvgNonSalaryBenefitsImportance': 99,
    'StackOverflowVisit': 'Prefer not to say',
    'StackOverflowHasAccount': "I'm not sure / I can't remember",
    'StackOverflowParticipate': 'Prefer not to say',
    'AIDangerous': 'Prefer not to say',
    'AIInteresting': 'Prefer not to say',
    'AIResponsible': 'Prefer not to say',
    'AIFuture': "I don't care about it, or I haven't thought about it.",
    'Benefit_SalaryBonuses': 99,
    'Benefit_RetirementPlan': 99,
    'Importance_Industry': 99,
    'Importance_RemoteWork': 99,
    'Importance_Technologies': 99
}

# Reemplazo de valores nulos en df_relevant
df_relevant.fillna(value=na_replacements, inplace=True)

C:\Users\Nicolas Cuaran\AppData\Local\Temp\ipykernel_7276\3689833266.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_relevant.fillna(value=na_replacements, inplace=True)


In [11]:
# Check for remaining nulls
null_counts = df_relevant.isnull().sum().head(41)
print('Remaining null counts:\n', null_counts[null_counts > 0])

Remaining null counts:
 Series([], dtype: int64)


In [12]:
# Check for duplicates
duplicates = df_relevant.duplicated().sum()
if duplicates > 0:
    print(f'Warning: {duplicates} duplicate rows found.')

# Drop duplicates
df_relevant.drop_duplicates(inplace=True)
print(f'Number of duplicate rows: {duplicates}')

Number of duplicate rows: 3328


C:\Users\Nicolas Cuaran\AppData\Local\Temp\ipykernel_7276\1508723717.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_relevant.drop_duplicates(inplace=True)


In [13]:
df_relevant.duplicated().sum()

np.int64(0)

In [14]:
df_relevant.head(1)

,Country,Gender,Age,FormalEducation,RaceEthnicity,DevType,CompanySize,Employment,YearsCodingProf,UndergradMajor,...,AIInteresting,AIResponsible,AIFuture,Hobby,Benefit_SalaryBonuses,Benefit_RetirementPlan,Importance_Industry,Importance_RemoteWork,Importance_Technologies,StandardizedMonthlySalaryCOP
0,Kenya,Male,25 - 34 years old,"Bachelor’s degree (BA, BS, B.Eng., etc.)",Black or of African descent,Full-stack developer,20 to 99 employees,Employed part-time,3-5 years,Mathematics or statistics,...,Algorithms making important decisions,The developers or the people creating the AI,I'm excited about the possibilities more than ...,Yes,99.0,99.0,10.0,3.0,1.0,0.0


In [15]:
df_relevant.info(41)

<class 'pandas.core.frame.DataFrame'>
Index: 95527 entries, 0 to 98851
Data columns (total 41 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Country                         95527 non-null  object 
 1   Gender                          95527 non-null  object 
 2   Age                             95527 non-null  object 
 3   FormalEducation                 95527 non-null  object 
 4   RaceEthnicity                   95527 non-null  object 
 5   DevType                         95527 non-null  object 
 6   CompanySize                     95527 non-null  object 
 7   Employment                      95527 non-null  object 
 8   YearsCodingProf                 95527 non-null  object 
 9   UndergradMajor                  95527 non-null  object 
 10  UpdateCV                        95527 non-null  object 
 11  LanguageWorkedWith              95527 non-null  object 
 12  DatabaseWorkedWith              95527

## Table Creation

We define and create the `clean_survey` table in PostgreSQL with appropriate data types for each column.

In [16]:
# Load data into clean_survey table

from sqlalchemy import text  # Import text for raw SQL

try:
    # Load DataFrame into clean_survey
    df_relevant.to_sql(
        'clean_survey',
        engine,
        if_exists='replace',  # Replace table if it exists
        index=False,          # Do not write DataFrame index
        method='multi',       # Use multi-row inserts for performance
        chunksize=1000        # Process in batches of 1000 rows
    )
    
    # Verify row count in database
    with engine.connect() as connection:
        row_count = connection.execute(text("SELECT COUNT(*) FROM clean_survey")).scalar()
        print(f"Cleaned data loaded successfully. {row_count} rows inserted into clean_survey.")

except Exception as e:
    print(f"Error loading data into clean_survey: {e}")
    raise

Cleaned data loaded successfully. 95527 rows inserted into clean_survey.


## Verification

We query the first 5 rows of the `clean_survey` table to confirm the data was loaded correctly.

In [17]:
print(f"Rows in df_clean: {len(df_relevant)}")
# Luego del to_sql
print(f"Rows in database: {row_count}")
assert len(df_relevant) == row_count, "Row counts do not match!"

Rows in df_clean: 95527
Rows in database: 95527
